# Week 9 Lab：Flow maps 與 MeanFlow identity

## 學習目標
- 數值驗證 flow map 的 identity 與 semigroup 性質。
- 比較瞬時速度（切線）與平均速度（弦）。
- 驗證 MeanFlow identity，並比較一步平均速度與多步 Euler。

> **誠實註記**：本 notebook 使用解析平移 flow；`mean_velocity` 是公式直接計算的 oracle，不是訓練好的 MeanFlow checkpoint。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 909
rng = np.random.default_rng(SEED)

def displacement(t):
    t = np.asarray(t)
    return np.stack([1.8 * t, .9 * np.sin(np.pi * t) + .4 * t], axis=-1)

def velocity(t):
    t = np.asarray(t)
    return np.stack([np.ones_like(t) * 1.8, .9 * np.pi * np.cos(np.pi * t) + .4], axis=-1)

def flow_map(x, t, s):
    return np.asarray(x) + displacement(s) - displacement(t)

x = rng.normal(size=(500, 2))
t, r, s = .17, .56, .91
direct = flow_map(x, t, s)
composed = flow_map(flow_map(x, t, r), r, s)
print('identity error:', np.max(np.abs(flow_map(x, t, t) - x)))
print('semigroup error:', np.max(np.abs(direct - composed)))

In [ ]:
def mean_velocity(t, r):
    if np.isclose(t, r):
        return velocity(t)
    return (displacement(r) - displacement(t)) / (r - t)

def d_mean_dt(t, r, h=1e-5):
    return (mean_velocity(t + h, r) - mean_velocity(t - h, r)) / (2 * h)

pairs = [(.05, .95), (.2, .6), (.48, .52)]
for t, r in pairs:
    u = mean_velocity(t, r)
    rhs = velocity(t) + (r - t) * d_mean_dt(t, r)
    print(f'(t,r)=({t:.2f},{r:.2f})  identity error={np.linalg.norm(u-rhs):.2e}')

fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), constrained_layout=True)
origin = np.zeros(2)
for ax, (t, r) in zip(axes, pairs):
    v = velocity(t)
    u = mean_velocity(t, r)
    corr = (r - t) * d_mean_dt(t, r)
    ax.quiver(*origin, *v, angles='xy', scale_units='xy', scale=1, color='tab:blue', label='v(t)')
    ax.quiver(*origin, *u, angles='xy', scale_units='xy', scale=1, color='tab:orange', label='u(t,r)')
    ax.quiver(*v, *corr, angles='xy', scale_units='xy', scale=1, color='tab:red', label='correction')
    ax.set(xlim=(-.2, 2.2), ylim=(-3.2, 3.2), aspect='equal', title=f't={t:.2f}, r={r:.2f}')
axes[0].legend(fontsize=8)
plt.show()

In [ ]:
def euler_sample(x0, n_steps):
    x = np.array(x0, copy=True)
    dt = 1 / n_steps
    for k in range(n_steps):
        x = x + dt * velocity(k * dt)
    return x

source = rng.normal(size=(1000, 2))
truth = flow_map(source, 0, 1)
meanflow_one_step = source + mean_velocity(0, 1)
steps = np.array([1, 2, 4, 8, 16, 32, 64])
euler_error = np.array([np.sqrt(np.mean((euler_sample(source, n) - truth) ** 2)) for n in steps])
meanflow_error = np.sqrt(np.mean((meanflow_one_step - truth) ** 2))
print('oracle MeanFlow one-step RMSE:', meanflow_error)
fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(steps, euler_error, 'o-', label='instantaneous velocity + Euler')
ax.axhline(max(meanflow_error, 1e-16), color='tab:orange', ls='--', label='oracle mean velocity, one step')
ax.set(xlabel='function evaluations', ylabel='endpoint RMSE', title='Average velocity integrates the whole interval')
ax.legend()
ax.grid(True, which='both', alpha=.25)
plt.show()

In [ ]:
starts = rng.normal(size=(10, 2))
ts = np.linspace(0, 1, 120)
fig, ax = plt.subplots(figsize=(6, 5))
for start in starts:
    path = np.array([flow_map(start, 0, t) for t in ts])
    ax.plot(path[:, 0], path[:, 1], alpha=.7)
    chord = np.vstack([start, flow_map(start, 0, 1)])
    ax.plot(chord[:, 0], chord[:, 1], color='0.55', ls=':', lw=1)
ax.set(aspect='equal', title='Curved trajectory versus its MeanFlow chord')
plt.show()

## 讀者練習 / TODO
把 `displacement` 的第二維改成更高頻的 $\sin(3\pi t)$。比較 Euler 曲線與 oracle MeanFlow；再用有限差分步長 `h=1e-2,1e-4,1e-6` 驗證 MeanFlow identity 的數值誤差。